# The Evaluator–Optimizer Pattern in LangGraph
### A self-improving LLM loop: generate → critique → refine → repeat

**A note before we start:** you mentioned "N scripts" but only one file came through —
`evaluator_optimizer.py`. So this notebook is a single continuous lesson built entirely
around that one script, reordered and unpacked into the sequence that teaches it best.
Everything here is real code from your file, lightly cleaned up for narrative flow.

---

## The problem this pattern solves

If you ask an LLM to write a product description in one shot, you get *one attempt at
quality*. There's no mechanism forcing it to notice its own weak points. Humans don't
write like this either — a first draft goes through a critic (an editor, a manager, an
internal voice), gets feedback, and gets revised. Often more than once.

The **evaluator-optimizer** pattern encodes exactly that workflow as a graph:

```
generate → evaluate → (good enough? stop : feedback → generate again)
```

Two LLM calls play two different roles:
- **Optimizer** — a *generator*. Its job is to produce something, and later, to revise
  it based on criticism.
- **Evaluator** — a *critic*. Its only job is to judge the optimizer's output against
  explicit criteria and hand back structured, actionable feedback.

This is one of the canonical "agentic workflow" patterns (alongside things like
routing, prompt chaining, and parallelization) because it doesn't rely on the LLM
"trying harder" — it relies on **explicit iteration with a stopping condition**, which
is something loops are good at and single LLM calls are not.

You already know LangGraph's core building blocks (`StateGraph`, nodes, edges) from
your other notebooks, so we won't re-derive those from scratch. What's genuinely new
and worth slowing down for here is: (1) how **structured output via Pydantic** turns
free-text LLM responses into typed, validated objects that can be programmatically
compared and fed back into prompts, and (2) how the **conditional edge** encodes a
loop with a stop condition inside a graph that is otherwise acyclic-looking.

Let's build it piece by piece.

## Step 1 — Imports and environment

Nothing exotic here, but worth being precise about *why* each import exists, since
every piece will get used for a specific reason later.

In [ ]:
from langchain_groq import ChatGroq          # Groq-hosted chat models via LangChain's chat model interface
from dotenv import load_dotenv                # loads GROQ_API_KEY etc. from a .env file into os.environ
from langgraph.graph import StateGraph, START, END   # the graph builder + sentinel start/end nodes
from typing import TypedDict, List, Literal   # typed state + a type that restricts a variable to specific string values
from pydantic import BaseModel, Field         # for structured, validated LLM outputs

load_dotenv()

**Why each of these:**

- `ChatGroq` — LangChain's wrapper around Groq's API. You get the same `.invoke()`,
  `.with_structured_output()`, streaming, etc. interface as any other LangChain chat
  model, so the rest of the code doesn't care that it's Groq specifically underneath.
- `load_dotenv()` — reads a `.env` file (containing e.g. `GROQ_API_KEY=...`) and injects
  those values into `os.environ`. `ChatGroq` then picks the key up automatically from
  the environment — you never hardcode it in the script.
- `TypedDict` — lets you declare the *shape* of a dict with named, typed keys, without
  it being a full class with runtime enforcement. LangGraph uses this shape to know
  what keys exist on the graph's state and validate that nodes return dicts with a
  subset of those keys.
- `Literal` — restricts a variable/return type to an exact, closed set of string (or
  other literal) values. We'll use it in a moment to say "this function can only ever
  return the string `'optimizer'` or the string `'end'` — nothing else."
- `BaseModel`, `Field` — Pydantic's building blocks for structured output. More on this
  below, since it's central to the whole pattern.

## Step 2 — Designing the graph's state

Every LangGraph node reads from and writes to a shared **state** object. You define its
shape once, up front, as a `TypedDict`. Think of this as the "memory" that flows through
every node in the graph — each node gets the whole current state as input, and returns
a *partial* dict of the keys it wants to update.

In [ ]:
class OptimizationState(TypedDict):
    product_name        : str
    product_features    : List[str]
    target_audience     : str
    current_description : str
    evaluation_result   : dict
    feedback            : str
    iteration_count     : int
    max_iterations      : int
    is_approved         : bool
    iteration_history   : List[dict]

A few design choices worth noticing:

- **`current_description`** is the single field both nodes read *and* write. The
  optimizer writes a new draft into it; the evaluator reads it to judge it. This is
  the state field that "loops."
- **`iteration_count` / `max_iterations`** together form the loop's stopping condition
  — one counts up, the other is a fixed ceiling. This is the classic "budget" pattern
  for bounding an otherwise-unbounded agentic loop, so a stubborn evaluator can't send
  you into an infinite (and infinitely billed) loop.
- **`iteration_history`** is an accumulator — every pass through the evaluator appends
  a record to it, giving you a full audit trail of every draft and its scores. This is
  *not* just for humans afterward — it's exactly the kind of thing you'd log to
  LangSmith or a DB in production, or show as a UI timeline.
- **`evaluation_result`** is typed as a plain `dict` here rather than a Pydantic model.
  That's a deliberate simplification — the *rich* validated object lives only inside
  the evaluator node; once it leaves the node, it gets flattened to a dict via
  `.model_dump()`, because `TypedDict` fields are meant to be plain JSON-serializable
  data, not live Pydantic objects (that keeps state easy to checkpoint / serialize if
  you ever add a checkpointer, which you already know from your LangGraph work).

⚠️ **Style nit, not a bug:** the spacing around `:` is inconsistent (`is_approved` has
two spaces, `iteration_history` has four). Cosmetic only — Python doesn't care — but
worth normalizing if you run this through a formatter like `black`.

## Step 3 — Structured output: making the LLM return *objects*, not paragraphs

Here's the core tension the whole pattern needs to resolve: an evaluator that returns
free text like *"pretty good, maybe improve the CTA"* is useless to a program. You can't
do `if score >= 8` on a paragraph. You need the LLM's judgement expressed as **typed,
validated fields** you can branch on in code.

This is what `.with_structured_output()` + a Pydantic model gives you. Under the hood,
LangChain converts your Pydantic class into a JSON Schema, sends that schema to the
model as part of the request (with Groq, via the `method="json_schema"` argument you'll
see in a moment), and the model is constrained to return JSON matching that schema.
LangChain then parses that JSON straight into an instance of your Pydantic class — so
`evaluation.overall_score` is a real Python `int`, not a string you have to `int()` and
hope doesn't crash.

In [ ]:
class ProductDescription(BaseModel):
    headline        : str = Field(description="Catchy headline (max 10 words)")
    description     : str = Field(description="Main product descripion (100-150 words)")
    key_benefits    : List[str] = Field(description="3-5 key benefits as bullet points.")
    call_to_action  : str = Field(description="Compelling call-to-action")


class Evaluation(BaseModel):
    overall_score       : int = Field(description="Overall quality score 1-10", ge=1, le=10)
    clarity_score       : int = Field(description="Clarity score 1-10", ge=1, le=10)
    persuasiveness_score: int = Field(description="Persuasiveness score 1-10", ge=1, le=10)
    audience_fit_score  : int = Field(description="Target audience fit (1-10)", ge=1, le=10)
    is_approved          : bool = Field(description="Whether description meets standards (score>=8)")
    strengths            : List[str] = Field(description="What works well")
    weaknesses            : List[str] = Field(description="What needs improvement")
    specific_feedback    : str = Field(description="Detailed, actionable feedback for revision")

**What's happening field by field:**

- Every `Field(description=...)` string isn't just documentation for *you* — it gets
  embedded into the JSON Schema sent to the model. The model literally reads
  `"Overall quality score 1-10"` as an instruction for what to put there. Sloppy
  descriptions produce sloppy outputs, so these strings are prompt-engineering, not
  just comments.
- **`ge=1, le=10`** — Pydantic validators: *greater-than-or-equal* to 1, *less-than-
  or-equal* to 10. If the model somehow returns `12`, Pydantic raises a
  `ValidationError` when parsing the response, rather than silently letting a garbage
  score flow into your state. This is your safety net against the model not perfectly
  following the schema.
- `key_benefits: List[str]` — Pydantic knows how to validate a JSON array of strings
  into a Python `list[str]` automatically; you don't write any parsing code for this
  yourself.

⚠️ **Bug watch — typo in the schema text, not just cosmetic:** `Field(description=
"Main product descripion (100-150 words)")` — "descripion" is missing a *t*. Since this
description is sent to the model as an instruction, it's harmless here (the model will
still understand it), but it's a good habit to proofread `Field` descriptions carefully
precisely *because* they're read by the model, not just by you.

⚠️ **Design risk worth flagging (not a syntax bug, a logic one):** `is_approved` is a
field the LLM fills in itself, described as "score >= 8." But nothing in the schema
*enforces* that `is_approved == (overall_score >= 8)`. Pydantic validates each field
independently — it has no idea these two fields are supposed to agree. In practice an
LLM could return `overall_score=6, is_approved=True` and Pydantic would accept it
happily, because both values are individually valid. If you want that invariant
actually enforced, you'd add a Pydantic **model validator** (`@model_validator`) that
cross-checks the fields after parsing. Something to file away for when you extend this.

## Step 4 — Two LLMs, two roles

Nothing stops you from using one model for both jobs, but there's a real reason to use
two here.

In [ ]:
llm = ChatGroq(model='openai/gpt-oss-20b')       # the optimizer / generator
eval_llm = ChatGroq(model="openai/gpt-oss-120b")  # the evaluator / critic

The optimizer uses the smaller **20B** model — generation is cheap, fast, and
creative work, and you'll be calling it repeatedly across iterations, so latency/cost
matter more here. The evaluator uses the larger **120B** model — judgement calls
(is this actually persuasive? does it fit the audience?) benefit from a stronger model,
and you only call it once per iteration, so the extra cost is easy to justify.

This "cheap generator, strong critic" split is a common cost/quality lever in
multi-agent systems — you've probably seen the same idea in RL as *asymmetric* actor
vs. critic capacity. It's the same intuition here, just with prompted LLMs instead of
trained networks.

## Step 5 — The optimizer node: generate, then revise

This is the first **node** — a plain Python function that takes the current state in
and returns a partial state update out. LangGraph will call this function every time
control passes to the `"optimizer"` node in the graph.

The function branches on `iteration == 1`: the *first* time it runs, there's nothing to
revise yet, so it writes a fresh description from the product info alone. On every
later call, it's working off `state['current_description']` plus the evaluator's
`state['feedback']` and `state['evaluation_result']` — i.e. it's doing **feedback-
conditioned regeneration**, not generating from scratch again.

In [ ]:
def generate_description(state: OptimizationState) -> OptimizationState:

    iteration = state['iteration_count']

    print(f"\n{'='*70}")
    print(f'OPTIMIZER: Iteration {iteration}')

    optimizer_llm = llm.with_structured_output(ProductDescription, method="json_schema")

    if iteration == 1:
        print("Creating initial product description ...")

        prompt = f"""Create a compelling product description for:

        Product: {state['product_name']}
        Features: {', '.join(state['product_features'])}
        Target Audience: {state['target_audience']}

    Requirements:
    - Headline : Catchy and concise (max 10 words)
    - Description: Engaging and informative (100-150 words)
    - Key Benefits: 3-5 clear, compelling benefits
    - Call-to-action: Strong, action-oriented CTA

    Make it persuasive and tailored to the target audience.
    """
    else:
        print("Refining description based on evaluation feedback...\n")

        prompt = f"""Improve this product description based on feedback:

Product: {state['product_name']}
Target Audience: {state['target_audience']}

CURRENT DESCRIPTION:
{state['current_description']}

EVALUATION SCORES:
- Overall: {state['evaluation_result'].get('overall_score', 0)}/10
- Clarity: {state['evaluation_result'].get('clarity_score', 0)}/10
- Persuasiveness: {state['evaluation_result'].get('persuasiveness_score', 0)}/10
- Audience Fit: {state['evaluation_result'].get('audience_fit_score', 0)}/10

FEEDBACK TO ADDRESS:
{state['feedback']}

CRITICAL: Focus on the specific weaknesses mentioned. Make targeted improvements to:
1. Address each point in the feedback
2. Maintain the strengths that were working
3. Increase scores in weak areas

Generate an improved version that addresses all feedback
"""

    description = optimizer_llm.invoke(prompt)

    formatted_description = f"""HEADLINE: {description.headline}

DESCRIPTION:
{description.description}

KEY BENEFITS:
{chr(10).join([f"* {benefit}" for benefit in description.key_benefits])}

CALL TO ACTION:
{description.call_to_action}
"""
    print("GENERATED Description:")
    print('-'*70)
    print(formatted_description + '\n')

    return {
        'current_description': formatted_description,
        'iteration_count'    : iteration + 1
    }

**Function-by-function breakdown:**

- **Purpose:** produce (or revise) a `ProductDescription`, format it into a readable
  block of text, and hand back a state update.
- **Inputs:** the full `OptimizationState` dict (LangGraph always passes the whole
  current state to every node — the function just reads the keys it needs).
- **Outputs:** a *partial* dict — only `current_description` and `iteration_count` are
  returned. LangGraph merges this into the existing state; you don't need to return
  every key, only the ones you're changing.
- **Internal logic:** branch on `iteration == 1` to choose a "create" vs. "revise"
  prompt → call the structured-output LLM → reformat the typed `ProductDescription`
  object back into a single human-readable string for storage/printing/re-prompting.
- **Methods used:**
  - `llm.with_structured_output(Model, method="json_schema")` — wraps the base chat
    model so that `.invoke()` returns a parsed Pydantic instance instead of a raw
    `AIMessage`. `method="json_schema"` tells LangChain which structured-output
    mechanism the underlying provider supports (Groq exposes an OpenAI-compatible
    JSON-schema-constrained decoding mode; other providers might use tool-calling
    under the hood instead — LangChain abstracts that choice away, `method` just picks
    the strategy explicitly).
  - `', '.join(state['product_features'])` — turns the feature list into a single
    comma-separated string for the prompt (an LLM prompt is just text; you always have
    to flatten structured Python data into a string before it reaches the model).
  - `chr(10)` — this is just `'\n'` written as a character code. It's used inside the
    f-string's `.join()` call because, pre-Python-3.12, you **cannot** put a backslash
    (like `'\n'`) directly inside an f-string expression `{...}` — `chr(10)` is the
    classic workaround.
- **Possible errors:** if the model fails to return valid JSON matching the schema (a
  rare but real occurrence, especially with smaller/faster models), `.invoke()` raises
  a validation/parsing exception — nothing in this function catches it, so it would
  propagate up and crash the graph run. Something to consider wrapping in a `try/except`
  with a retry once you're running this unattended.

⚠️ **Bug watch — real syntax bugs in the original:**

1. `print(f"\n{"="*70}")` — the original script nests **double quotes inside double
   quotes** inside the same f-string. This only works on **Python 3.12+**, where PEP
   701 relaxed f-string quoting rules. On any earlier Python (3.11 and below), this is
   a flat-out `SyntaxError`. I switched the inner quotes to single quotes above
   (`f"\n{'='*70}"`) so the cell runs on any Python version — worth checking
   `python --version` in your environment if you hit a surprise `SyntaxError` running
   the original file as-is.
2. The original prompt text says `"Create a compelling product for descrition for:"` —
   garbled wording ("for descrition for") plus a typo (descrition → description). I
   cleaned this to `"Create a compelling product description for:"` above. Not a
   crash-causing bug, but it's the kind of thing that quietly degrades prompt quality.
3. `"Headline : Catcy and concise"` — "Catcy" → "Catchy", fixed above.
4. `"Key Benfits: 3-5 clear..."` — "Benfits" → "Benefits", fixed above.
5. A smaller one: `optimizer_llm` is rebuilt with `.with_structured_output(...)` on
   *every single call* to this node, even though `ProductDescription` never changes.
   Not incorrect, just slightly wasteful — you could build it once at module level
   next to `llm = ChatGroq(...)` instead.

## Step 6 — The evaluator node: score it, and remember scoring it

Structurally this mirrors the optimizer node — build a structured-output LLM, prompt
it, get back a typed object — but its job is judgement, not creation, and it has one
extra responsibility the optimizer doesn't: **appending to the audit trail**
(`iteration_history`).

In [ ]:
def evaluate_description(state: OptimizationState) -> OptimizationState:

    print(f"\n{'='*70}")
    print('EVALUATOR: Reviewing Description')
    print(f"{'='*70}")

    evaluator = eval_llm.with_structured_output(Evaluation, method="json_schema")

    prompt = f"""Evaluate this product description objectively:

Product: {state['product_name']}
Target Audience: {state['target_audience']}

DESCRIPTION TO EVALUATE:
{state['current_description']}

Evaluate on these criteria (1-10 scale):
1. CLARITY: Is it clear and easy to understand?
2. PERSUASIVENESS: Does it effectively sell the product?
3. AUDIENCE FIT: Does it resonate with the target audience?

APPROVAL CRITERIA: Overall score must be 8 or higher to approve.

Provide:
- Scores for each criterion
- Overall score (average of criteria)
- Whether it's approved (score >= 8)
- Specific strengths (what's working well)
- Specific weaknesses (what needs improvement)
- Actionable feedback for the next iteration

Be objective and constructive."""

    evaluation = evaluator.invoke(prompt)

    print("Evaluation Results")
    print('-'*70)
    print(f"Overall Score: {evaluation.overall_score}/10")
    print(f"Clarity Score: {evaluation.clarity_score}/10")
    print(f"Persuasiveness Score: {evaluation.persuasiveness_score}/10")
    print(f"Audience Fit: {evaluation.audience_fit_score}/10")
    print(f"STATUS: {'APPROVED' if evaluation.is_approved else 'NEEDS REVISION'}")

    print("\nStrengths:")
    for strength in evaluation.strengths:
        print(f"- {strength}")

    print("\nWeaknesses:")
    for weakness in evaluation.weaknesses:
        print(f"- {weakness}")

    print(f"\nFeedback: {evaluation.specific_feedback}")

    iteration_record = {
        "iteration": state["iteration_count"],
        "description": state["current_description"],
        "scores": {
            "overall": evaluation.overall_score,
            "clarity": evaluation.clarity_score,
            "persuasiveness": evaluation.persuasiveness_score,
            "audience_fit": evaluation.audience_fit_score
        },
        "approved": evaluation.is_approved,
        "feedback": evaluation.specific_feedback
    }

    history = state.get("iteration_history", [])
    history.append(iteration_record)

    return {
        "evaluation_result": evaluation.model_dump(),
        "feedback": evaluation.specific_feedback,
        "is_approved": evaluation.is_approved,
        "iteration_history": history
    }

**Function-by-function breakdown:**

- **Purpose:** score the current draft against fixed criteria and produce feedback the
  optimizer can act on next round.
- **Inputs:** full `OptimizationState`; only reads `current_description`,
  `product_name`, `target_audience`, `iteration_count`, `iteration_history`.
- **Outputs:** partial state update with `evaluation_result`, `feedback`,
  `is_approved`, and the growing `iteration_history`.
- **Internal logic:** prompt the evaluator LLM for a structured `Evaluation` → log it
  to console → build a plain-dict `iteration_record` (a *snapshot*, decoupled from the
  live Pydantic object) → append that record to the running history list → return
  everything the router / next optimizer call will need.
- **Methods used:**
  - `evaluation.model_dump()` — Pydantic's method to convert a `BaseModel` instance
    into a plain `dict` (recursively, for nested models/lists too). This is why
    `evaluation_result` in `OptimizationState` is typed as `dict`, not `Evaluation` —
    once it's stored in state, it's meant to be plain data, not a live object.
  - `state.get("iteration_history", [])` — using `.get()` with a default here is
    defensive: if `iteration_history` were ever missing from state, this falls back to
    a fresh list instead of a `KeyError`. In this script it's always present in the
    initial state, so the default rarely triggers, but it's a good habit for any node
    that reads keys another part of the graph is responsible for initializing.
- **Possible errors:** same structured-output parsing risk as the optimizer. Also — see
  bug watch below — a subtler issue with *how* the history list is being mutated.

⚠️ **Bug watch:**

1. Same f-string quoting issue as the optimizer (`f"\n{"="*70}"` needs Python 3.12+;
   fixed above to use single quotes inside).
2. **The original script never prints the "Weaknesses" header** before its weaknesses
   loop — it prints `Strengths:`, loops strengths, then goes straight into the
   weaknesses loop with no `print(f"\nWeaknesses:")` line first. Not a crash, just an
   output-formatting gap; I added the missing header above.
3. **Mutable-state subtlety, worth knowing even though it "works" here:**
   `history = state.get("iteration_history", [])` fetches the *same list object*
   already sitting in state, and `history.append(...)` mutates it **in place** before
   it's returned as the new value for that key. In a simple, single-branch graph like
   this one, that's harmless — there's only ever one path through the graph at a time.
   But it's a pattern to be wary of: if you ever parallelize this (e.g. evaluate
   multiple draft variants concurrently, a pattern you've explored elsewhere), mutating
   a shared list in place from multiple branches becomes a race condition. The safer
   habit is `history = state.get("iteration_history", []) + [iteration_record]`, which
   builds a *new* list instead of mutating the old one.

## Step 7 — The routing function: turning a state check into a loop

Everything so far has been two straight-line nodes. The actual *loop* — "keep
regenerating until approved or out of budget" — lives entirely in one function that
LangGraph calls a **conditional edge**: a plain function that inspects state and
returns the *name* of the next node to go to.

In [ ]:
def should_continue(state: OptimizationState) -> Literal['optimizer', 'end']:

    if state['is_approved']:
        print("\nSUCCESS: Description Approved")
        return 'end'

    if state['iteration_count'] > state['max_iterations']:
        print(f"\nMAX ITERATIONS REACHED: Stopping at iteration {state['iteration_count']-1}")
        return 'end'

    print(f"\nCONTINUING: Routing back to optimizer for iteration {state['iteration_count']}")
    return 'optimizer'

**Function-by-function breakdown:**

- **Purpose:** decide, after every evaluation, whether the loop stops or continues.
- **Inputs:** state — specifically `is_approved`, `iteration_count`, `max_iterations`.
- **Outputs:** a plain **string** — not a state update. Conditional-edge functions in
  LangGraph don't modify state at all; they only return a routing key.
- **Internal logic (priority order matters here):**
  1. If the evaluator approved it → stop, we're done, success case.
  2. *Otherwise*, if we've already used up the iteration budget → stop anyway, but as a
     "gave up" case rather than a success case.
  3. Otherwise → go back to the optimizer with the feedback that's now sitting in state.
- **`Literal['optimizer', 'end']`** — this returns type annotation isn't just
  documentation. LangGraph reads it (or the `path_map` dict you'll see next) to know
  every valid destination this router can send control to — it's how the graph
  visualizer and validation logic know the full set of possible edges out of this node,
  even though only one branch executes at runtime.
- **Possible errors:** none inherent to this function, but note the ordering — because
  `is_approved` is checked *first*, an approved-but-out-of-budget draft (edge case:
  approved exactly on the last allowed iteration) still correctly reports as `SUCCESS`,
  not `MAX ITERATIONS REACHED`. That's a deliberate, correct choice in the ordering —
  worth noticing *why* it's ordered this way, since swapping the two checks would
  silently misreport a perfectly fine approved result as a failure.

⚠️ **Bug watch — this one is a genuine crash risk, not just style:**

The original line was:

```python
print(f'\n CONTINUING: Routing back to optimizer for iteration {state['iteration_count']}')
```

This uses a **single-quoted f-string containing a single-quoted dict access
(`state['iteration_count']`)** — the *exact same* quote character, nested. Unlike the
double-quote case earlier (which at least has a Python-3.12+ escape hatch), reusing the
identical quote character to open and close both the outer string and the inner
expression is ambiguous to the parser in a way that's a hard `SyntaxError` in every
Python version, including 3.12+. This line cannot run as written, period — it needs
either `state["iteration_count"]` (different quote type inside) or an f-string built
with different outer/inner quote characters, as fixed above.

## Step 8 — Wiring it all into a graph

You've already worked with `StateGraph` in your other notebooks, so this part is
mostly familiar — the interesting bit is specifically **how a conditional edge encodes
a cycle** in what would otherwise look like a straight-line pipeline.

In [ ]:
builder = StateGraph(OptimizationState)

builder.add_node('optimizer', generate_description)
builder.add_node('evaluator', evaluate_description)

builder.add_edge(START, "optimizer")
builder.add_edge("optimizer", "evaluator")

builder.add_conditional_edges(
    "evaluator",
    should_continue,
    {
        "optimizer": "optimizer",
        "end": END
    }
)

graph = builder.compile()

**What each call does:**

- `StateGraph(OptimizationState)` — creates the graph builder, telling it the shape of
  state every node will read/write (this is where your `TypedDict` from Step 2 gets
  used).
- `add_node(name, function)` — registers a node under a string name, bound to the
  Python function that implements it.
- `add_edge(START, "optimizer")` — a **fixed, unconditional** edge: every run always
  begins at the optimizer. `START` is a LangGraph sentinel, not a real node.
- `add_edge("optimizer", "evaluator")` — another fixed edge: every generated draft
  always gets evaluated, no branching here.
- `add_conditional_edges("evaluator", should_continue, {...})` — this is the one doing
  real work. After `"evaluator"` runs, LangGraph calls `should_continue(state)`, gets
  back a string (`'optimizer'` or `'end'`), and looks that string up in the `path_map`
  dict (`{"optimizer": "optimizer", "end": END}`) to find the *actual* next node. The
  dict's keys must exactly match what `should_continue` can return — this is the same
  contract the `Literal['optimizer', 'end']` type hint was documenting.
- The **cycle** in this graph is exactly the edge `"evaluator" → "optimizer"` that
  `add_conditional_edges` can produce. Visually this graph is not a DAG (directed
  *acyclic* graph) at all — it has a real cycle — and that's fine; LangGraph is built
  to support cycles specifically for cases like this, unlike plain LCEL chains which
  are inherently linear/acyclic. This is one of the main reasons to reach for
  LangGraph over a plain `RunnableSequence` once your workflow needs to loop.
- `builder.compile()` — finalizes the graph into a runnable object. This is also the
  point where, if you wanted state persistence across runs, you'd pass a
  `checkpointer=...` here — you've used this before with `thread_id` configs in your
  other LangGraph work; this script simply doesn't need it since everything happens in
  one `invoke()` call.

**Why a graph instead of a plain Python `while` loop?** You *could* write this whole
thing as:
```python
while not approved and iteration <= max_iterations:
    draft = generate(...)
    evaluation = evaluate(draft)
    ...
```
and it would work. What the graph buys you instead: each node is independently
testable/swappable, the structure is *declarative* (you can literally draw it), you get
LangGraph's built-in streaming of intermediate states, and if you add a checkpointer you
get pause/resume and time-travel debugging for free. For a two-node loop like this the
`while` loop isn't unreasonable — the graph pays off more as the number of nodes and
branch conditions grows, or once you want to inspect/replay individual steps.

## Step 9 — Running the graph

`graph.invoke(initial_state)` runs the *entire* graph synchronously — following every
edge, including looping back through the conditional edge as many times as needed —
until it reaches `END`, then returns the final accumulated state as a plain dict.

Note every key of `OptimizationState` needs an initial value here, even the ones that
start "empty" (`""`, `{}`, `[]`, `False`) — LangGraph doesn't invent defaults for keys
you don't provide.

In [ ]:
print("="*70)
print('EVALUATOR-OPTIMIZER PATTERN: PRODUCT DESCRIPTION')
print("="*70)
print("Example: Fitness Tracker")
print("="*70)

result = graph.invoke({
    "product_name": "Fitpulse Pro Smart Watch",
    "product_features": [
        "Heart rate monitoring",
        "GPS tracking",
        "Sleep analysis",
        "Waterproof to 50m",
        "7-day battery life",
        "Smartphone notifications"
    ],
    "target_audience": "Health-conscious professionals aged 25-45",
    "current_description": "",
    "evaluation_result": {},
    "feedback": "",
    "iteration_count": 1,
    "max_iterations": 5,
    "is_approved": False,
    "iteration_history": []
})

print(f"\n{'='*70}")
print("FINAL APPROVED DESCRIPTION")
print(f"{'='*70}")
print(result['current_description'])

⚠️ **Bug watch (cosmetic):** the original banner prints
`"EVALUATORL-OPTIMIZER PATTERNL PRODUCT DESCRIPTION"` — a stray typo, "PATTERNL"
instead of "PATTERN" — fixed above. Purely a print statement, has zero effect on
behavior.

**Tracing the iteration budget once, since it's easy to get off-by-one here:**
`iteration_count` starts at `1`. Every optimizer call increments it by one
(`iteration + 1`), so after the *first* generate call it becomes `2`, and so on. The
router stops only when `iteration_count > max_iterations`. Working through it:
generate calls happen at counts `1, 2, 3, 4, 5` (five calls, matching `max_iterations
= 5`); after the 5th call, `iteration_count` becomes `6`, `6 > 5` is `True`, and the
loop halts on "MAX ITERATIONS REACHED" if it hasn't already been approved. So
`max_iterations=5` really does mean *at most 5 optimizer calls* — the off-by-one
doesn't bite here, but it's the kind of arithmetic worth tracing by hand once so you
trust it rather than assume it.

## Step 10 — Using the audit trail

The `result` dict is just the final `OptimizationState`. Because the evaluator kept
appending to `iteration_history`, you have a full record of every draft and how it
scored — useful for debugging *why* the loop stopped where it did, or for building a
UI that shows the revision history to a human.

In [ ]:
for record in result["iteration_history"]:
    scores = record["scores"]
    print(
        f"Iteration {record['iteration']}: "
        f"overall={scores['overall']}, approved={record['approved']}"
    )

print(f"\nTotal iterations run: {len(result['iteration_history'])}")
print(f"Final approval status: {result['is_approved']}")

**A debugging tip for this specific pattern:** if you ever find the loop running
all the way to `max_iterations` without ever approving, the *first* place to look is
`iteration_history` — specifically, whether the `overall` score is trending upward at
all across iterations. If it's flat or oscillating, that's a sign the optimizer isn't
actually incorporating `feedback` meaningfully (worth printing `state['feedback']`
right before the optimizer's revise-prompt is built, to confirm it's non-empty and
specific), rather than an issue with the graph wiring itself.

## Synthesis — how this connects to what you already know

- This is a **cyclic** LangGraph, unlike the linear LCEL chains and ReAct-style
  tool-calling agents you've built before. The mechanism that makes the cycle possible
  is `add_conditional_edges` mapping a router function's string output back to an
  *earlier* node — the same conditional-edges API you'd use for tool-routing agents,
  just pointed backward instead of forward.
- **Structured output via Pydantic** here plays exactly the role that a well-designed
  tool schema plays in a ReAct agent: it forces the model's response into something a
  program can branch on deterministically. The difference is intent — here it's used
  to get a *judgement* (scores, approval) rather than a *tool call*.
- The **two-model split** (cheap generator / stronger critic) is a cost-quality lever
  you'll see again anywhere you're running an LLM in a loop — the same reasoning
  applies if you ever build a self-critiquing RAG pipeline or an LLM-as-judge eval
  harness for your other projects.
- The **iteration-budget pattern** (`iteration_count` vs `max_iterations`, checked
  first-thing in the router) is the general-purpose way to bound *any* agentic loop —
  worth reusing verbatim any time you build something with an unbounded "keep trying"
  shape.

## Exercises

1. **Add a model validator.** Extend the `Evaluation` class with a Pydantic
   `@model_validator(mode="after")` that raises (or auto-corrects) if `is_approved`
   disagrees with `overall_score >= 8`. Re-run and see if it ever actually triggers.
2. **Fix the mutation risk.** Rewrite `evaluate_description` so `iteration_history` is
   built with `+ [iteration_record]` instead of in-place `.append()`, and confirm the
   graph still behaves identically.
3. **Add a third criterion.** Add an `originality_score` field to `Evaluation`, update
   the evaluator's prompt to ask for it, and fold it into `overall_score`'s definition
   in the prompt text. Notice how little else in the graph needs to change — that's the
   payoff of keeping node logic and graph wiring separate.
4. **Swap the loop for a `while` loop.** Rewrite the whole thing without LangGraph,
   using a plain Python loop with the same stopping logic, and compare how much
   boilerplate you save vs. lose. This is the fastest way to viscerally feel what the
   graph is (and isn't) buying you here.
5. **Make it parallel-safe.** Imagine generating 3 candidate descriptions per
   iteration and evaluating all 3 concurrently, keeping only the best. Sketch (in
   comments, doesn't need to run) what would need to change about state shape and the
   `iteration_history` mutation pattern to make that safe.

## Concepts covered

- Agentic workflow pattern: **generate → evaluate → conditionally loop**
- `TypedDict` for declaring LangGraph state shape
- Pydantic `BaseModel` + `Field(description=..., ge=..., le=...)` for structured,
  validated LLM output
- `llm.with_structured_output(Model, method="json_schema")`
- Using two differently-sized models for generator vs. critic roles
- Nodes as plain functions: full state in, partial state dict out
- `evaluation.model_dump()` — Pydantic model → plain dict, for state storage
- Conditional edges: `add_conditional_edges(node, router_fn, path_map)`
- `Literal[...]` type hints as a documented, checkable contract for router outputs
- Cycles in LangGraph vs. the inherently linear nature of LCEL chains
- Iteration-count/max-iteration budgeting as a general loop-termination pattern
- `chr(10)` as a `'\n'` workaround inside pre-3.12 f-string expressions
- Defensive `dict.get(key, default)` usage when reading shared state
- Why in-place mutation of a state field (e.g. `list.append`) is fine in a single
  linear run but risky once branches run concurrently